In [ ]:
import numpy as np
import pandas as pd
from numpy.linalg import norm
from tqdm.auto import tqdm

from libs.DataLoader import Loader
from libs.utils import constrained_top_selections, unconstrained_top_selection, NDCG
from libs.constants import *

In [ ]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

def similarities(v1, v_arr):
    return [cosine_similarity(v1, v2) for v2 in v_arr]

def weighted_avg(data, weights):
    return sum(w*v for w,v in zip(weights, data))/len(data)

def get_equal_weights(indices):
    n = len(indices)
    return np.ones(n) / n

def get_linear_weights(indices):
    n = len(indices)
    return (np.argsort(np.argsort(indices)) + 1) / (n * (n + 1) / 2)

def get_exponential_weights(indices):
    ranks = np.argsort(np.argsort(indices)) + 1
    return np.exp(ranks) / np.sum(np.exp(ranks))

In [ ]:
loader = Loader('ur0.01_ir0.01', content_embedding_size=64, batch_size=1_000_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=True)

In [ ]:
train_df: pd.DataFrame = train_df[train_df[TARGET] > 0]  # Interaction: positive
val_df: pd.DataFrame = val_df[val_df[TARGET] > 0]  # Interaction: positive

print(f"Train: {len(train_df)} Val: {len(val_df)}")

In [ ]:
agg = train_df \
    .merge(items_df[[ITEM, EMBEDDING]], on=ITEM) \
    .groupby(USER) \
    .agg({ITEM: list, EMBEDDING: list, TIME_INDEX: list})

val_items = pd.DataFrame({ITEM: val_df[ITEM].unique()})\
    .merge(items_df[[ITEM, EMBEDDING]], on=ITEM)

cold_val_items = set(val_items[ITEM].unique()) - set(train_df[ITEM].unique())

In [ ]:
scores = []
for item in tqdm(val_items.iloc, total=len(val_items)):
    scores.append(agg.apply(
        lambda row: weighted_avg(
            similarities(
                item[EMBEDDING],
                row[EMBEDDING]),
            get_linear_weights(row[TIME_INDEX])
        ),
        axis=1,
    ).rename(item[ITEM]))
scores = pd.concat(scores, axis=1)
scores.head(3)

In [ ]:
predict_unconstrained = unconstrained_top_selection(scores, 100)
predict_constrained = constrained_top_selections(scores, 100, 101)

In [ ]:
for pred, pred_name in [
    (predict_unconstrained, "unconstrained"),
    (predict_constrained, "constrained")
]:
    for true, true_name in [
        (val_df, "all"),
        (val_df[val_df[ITEM].isin(cold_val_items)], "cold")
    ]:
        print(f"Predict: {pred_name} Test: {true_name} NDCG: {NDCG(pred, true.groupby(ITEM).agg({USER: list})):.4f}")